# 보안 취약점 일일 브리핑 — LLM 오케스트레이션

[security_briefing_by_code.ipynb](security_briefing_by_code.ipynb) 는 **파이썬 코드가 순서를 정하는** 방식(Orchestrating by Code)이었습니다.
이 노트북은 같은 일을 [openai_agent_code.ipynb](openai_agent_code.ipynb) 의 **Step 8: 에이전트를 도구로** 방식으로 다시 만듭니다.

참고 노트북의 Sales Manager 는 세 개의 작성 에이전트를 **도구로** 불러 초안을 받고, 가장 좋은 것을 골라 발송 도구를 호출했습니다.
여기서는 **보안 브리핑 매니저** 가 자산 조회 · CVE 수집 · 매칭 · 브리핑 작성 · 메일 발송을 **도구로** 불러 일을 끝냅니다.

마지막 6단계에서는 참고 노트북 Step 9 의 또 다른 방식인 **핸드오프(Handoffs)** 로,
분석을 마친 매니저가 발송 업무를 다른 에이전트에게 **넘기는** 구조도 만들어 봅니다.

## by Code vs by LLM

| | by Code ([security_briefing_by_code](security_briefing_by_code.ipynb)) | by LLM (이 노트북) |
|---|---|---|
| 누가 순서를 정하나 | 파이썬 코드 (1→2→3→4→5 고정) | **매니저 에이전트**가 도구를 골라 호출 |
| 부품 | 함수와 에이전트 | 같은 부품을 **도구(tool)** 로 포장 |
| 요청이 바뀌면 | 코드를 고쳐야 함 | **지시문만 바꾸면 됨** |
| 예측 가능성 | 높음 | 낮음 → **안전장치는 코드에 둔다** |
| LLM 호출 수 | 3회 (매칭, 브리핑, 발송) | 2회 (매칭, 브리핑) + 매니저가 판단하는 횟수 |

## 구조

```
요청 ──▶ 보안 브리핑 매니저 (LLM) : 무엇을, 어떤 순서로, 몇 번 부를지 스스로 결정
            │
            ├─ list_assets          자산 목록 조회 — 파이썬 코드
            ├─ search_cves          제품별 CVE 후보 수집 — 파이썬 코드 (NVD API)
            ├─ match_cves           오탐 제거 — 자산 매칭 에이전트
            ├─ write_briefings      위험도 평가, 담당자별 정리 — 브리핑 작성 에이전트
            └─ send_briefing_email  담당자에게 메일 발송 — 파이썬 코드 (안전장치 포함)
```

모든 도구는 큰 데이터를 **작업 메모리(`state`)** 에 저장하고, 매니저에게는 **요약만** 돌려줍니다. (이유는 2단계에서 설명)

In [ ]:
import os
import json
import time
import smtplib
import threading
import requests
from collections import Counter
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage

from dotenv import load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, trace, function_tool, handoff, ModelSettings
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"
ASSETS_PATH = "assets.json"

# NVD API 키 (.env 의 NVD_KEY). 없어도 동작하지만 호출 제한이 10배 빡빡해집니다.
NVD_KEY = os.getenv("NVD_KEY")

# 호출 간격: 키가 있으면 30초당 50회, 없으면 30초당 5회
NVD_DELAY = 0.6 if NVD_KEY else 6.0

print(f"NVD API 키: {'사용' if NVD_KEY else '없음 (익명 호출)'} / 호출 간격 {NVD_DELAY}초")

## 1단계. 부품 준비 — security_briefing_by_code.ipynb 와 동일

아래 세 셀은 [security_briefing_by_code.ipynb](security_briefing_by_code.ipynb) 의 코드를 그대로 가져온 것입니다.
**부품은 바뀌지 않습니다. 바뀌는 것은 "누가 이 부품을 어떤 순서로 쓰는가" 뿐입니다.**

| 부품 | 역할 | 원래 노트북 |
|---|---|---|
| `load_assets`, `search_cves_for_product` | 자산 로드, NVD 조회 | 1~2단계 |
| `matcher`, `briefer` | 오탐 제거, 위험도 평가 에이전트 | 3~4단계 |
| `build_html`, `send_email` | 메일 본문 생성, 발송 | 5단계 |

In [ ]:
def load_assets():
    """자산 인벤토리를 읽어옵니다."""
    with open(ASSETS_PATH, encoding="utf-8") as f:
        return json.load(f)


def search_cves_for_product(product: str, days: int = 120, max_results: int = 4):
    """특정 제품과 관련된 최근 CVE를 NVD에서 조회합니다.

    Args:
        product: 제품 이름 (예: Apache Tomcat)
        days: 최근 며칠 이내에 공개된 것을 볼지
        max_results: 제품당 최대 건수
    """
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)

    # 키가 있으면 apiKey 헤더로 보낸다. 없으면 익명 호출(제한이 더 빡빡함).
    headers = {"apiKey": NVD_KEY} if NVD_KEY else {}

    r = requests.get(
        "https://services.nvd.nist.gov/rest/json/cves/2.0",
        params={
            "keywordSearch": product,
            "pubStartDate": start.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "pubEndDate": end.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "resultsPerPage": max_results,
            "noRejected": None,
        },
        headers=headers,
        timeout=30,
    )
    r.raise_for_status()

    result = []
    for item in r.json().get("vulnerabilities", []):
        cve = item["cve"]
        metrics = cve.get("metrics", {})
        # CVSS 버전이 여러 개일 수 있으므로 최신 것부터 찾는다
        key = next((k for k in ["cvssMetricV31", "cvssMetricV40", "cvssMetricV30", "cvssMetricV2"]
                    if k in metrics), None)
        cvss = metrics[key][0]["cvssData"] if key else {}

        result.append({
            "cve_id": cve["id"],
            "published": cve["published"][:10],
            "cvss_score": float(cvss.get("baseScore", 0.0)),
            "severity": cvss.get("baseSeverity", "UNKNOWN"),
            "description": cve["descriptions"][0]["value"][:400],
        })
    return result

In [ ]:
class MatchedCVE(BaseModel):
    cve_id: str
    asset_id: str
    reason: str          # 왜 영향을 준다고 판단했는지
    cvss_score: float
    summary_ko: str      # 취약점 내용 한글 요약


class ExcludedCVE(BaseModel):
    cve_id: str
    reason: str          # 왜 제외했는지


class MatchResult(BaseModel):
    matched: list[MatchedCVE]
    excluded: list[ExcludedCVE]


matcher = Agent(
    name="자산 매칭 에이전트",
    model=MODEL_NAME,
    instructions=(
        "당신은 취약점 관리 담당자입니다. 자산 목록과 CVE 후보 목록을 받습니다.\n"
        "각 CVE가 해당 자산에 '실제로' 영향을 주는지 판단하세요.\n"
        "- 키워드만 겹치고 다른 제품이면 제외하세요. "
        "(예: 자산은 OpenSSL인데 CVE는 rust-openssl 바인딩 취약점)\n"
        "- 설명에 명시된 영향 버전 범위가 자산 버전을 포함하지 않으면 제외하세요.\n"
        "- 버전 정보가 불명확하면 matched 에 넣되 reason에 '버전 확인 필요'라고 적으세요.\n"
        "제외한 것은 excluded 에 이유와 함께 넣으세요. reason은 한 문장으로 쓰세요."
    ),
    output_type=MatchResult,
)


class Advisory(BaseModel):
    cve_id: str
    asset_id: str
    asset_name: str
    severity: str        # 긴급 / 높음 / 보통 / 낮음
    summary_ko: str
    action: str          # 담당자가 바로 실행할 조치
    deadline: str        # 권고 기한


class OwnerBriefing(BaseModel):
    owner: str
    owner_email: str
    headline: str        # 메일 제목
    advisories: list[Advisory]


class BriefingResult(BaseModel):
    briefings: list[OwnerBriefing]


briefer = Agent(
    name="브리핑 작성 에이전트",
    model=MODEL_NAME,
    instructions=(
        "당신은 보안 어드바이저리 담당자입니다. 확정된 취약점 목록과 자산 정보를 받습니다.\n"
        "위험도(severity)는 CVSS 점수만이 아니라 자산의 중요도와 인터넷 노출 여부를 함께 고려해 "
        "'긴급' / '높음' / '보통' / '낮음' 중 하나로 정하세요.\n"
        "- 인터넷에 노출되고 중요도가 '상'인 자산은 한 단계 올립니다\n"
        "- 내부망 전용이면 한 단계 내릴 수 있습니다\n"
        "action에는 담당자가 바로 실행할 수 있는 구체적 조치를 1~2문장으로 쓰세요.\n"
        "deadline은 '24시간 이내', '1주일 이내' 처럼 적으세요.\n"
        "담당자(owner)별로 묶어서 출력하고, headline은 그 담당자에게 보낼 메일 제목입니다."
    ),
    output_type=BriefingResult,
)

In [ ]:
SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

# 데모 안전장치 1: 모든 메일을 본인 주소로 보냄. 실무에서는 None 으로 바꾼다.
ROUTE_ALL_TO_ME = EMAIL_ADDRESS

# 데모 안전장치 2: True 면 실제로 보내지 않고 "보냈다고 가정"만 한다.
DRY_RUN = False

SEVERITY_COLOR = {"긴급": "#c0392b", "높음": "#e67e22", "보통": "#2980b9", "낮음": "#7f8c8d"}


def build_html(briefing: OwnerBriefing) -> str:
    """담당자 한 명에게 보낼 HTML 메일 본문을 만듭니다."""
    rows = ""
    for a in briefing.advisories:
        color = SEVERITY_COLOR.get(a.severity, "#7f8c8d")
        rows += f"""
        <tr>
          <td style="padding:8px;border-bottom:1px solid #eee;">
            <span style="background:{color};color:#fff;padding:2px 8px;border-radius:3px;font-size:12px;">{a.severity}</span>
          </td>
          <td style="padding:8px;border-bottom:1px solid #eee;font-family:monospace;">{a.cve_id}</td>
          <td style="padding:8px;border-bottom:1px solid #eee;">{a.asset_name}</td>
          <td style="padding:8px;border-bottom:1px solid #eee;">{a.deadline}</td>
        </tr>
        <tr>
          <td></td>
          <td colspan="3" style="padding:0 8px 12px 8px;border-bottom:1px solid #eee;color:#555;font-size:13px;">
            {a.summary_ko}<br><b>조치:</b> {a.action}
          </td>
        </tr>"""

    return f"""
    <div style="font-family:'Malgun Gothic',sans-serif;max-width:720px;">
      <h2 style="border-bottom:2px solid #333;padding-bottom:8px;">보안 취약점 일일 브리핑</h2>
      <p>{briefing.owner}님, 담당 자산에서 조치가 필요한 취약점
         <b>{len(briefing.advisories)}건</b>이 확인되었습니다.</p>
      <table style="border-collapse:collapse;width:100%;font-size:14px;">
        <tr style="background:#f4f4f4;">
          <th style="padding:8px;text-align:left;">위험도</th>
          <th style="padding:8px;text-align:left;">CVE</th>
          <th style="padding:8px;text-align:left;">자산</th>
          <th style="padding:8px;text-align:left;">조치 기한</th>
        </tr>
        {rows}
      </table>
      <p style="color:#888;font-size:12px;margin-top:16px;">
        본 메일은 보안 브리핑 매니저 에이전트가 NVD 데이터를 기반으로 자동 생성했습니다. (수업용 데모)
      </p>
    </div>"""


def send_email(to_address: str, subject: str, html_body: str) -> str:
    """HTML 메일을 발송합니다."""
    msg = EmailMessage()
    msg["Subject"] = subject
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to_address
    msg.set_content("HTML 메일입니다. HTML을 지원하는 클라이언트에서 확인하세요.")
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP_SSL(SMTP_SERVER, 465) as smtp:
        smtp.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        smtp.send_message(msg)

    return f"발송 완료 -> {to_address}"

## 2단계. 작업 메모리 — 데이터는 코드가 들고 있는다

참고 노트북에서는 `as_tool` 한 줄로 에이전트를 도구로 바꿨습니다.

```python
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
```

`as_tool` 로 만든 도구는 매니저가 넘겨 주는 **문자열 하나(`input`)** 만 받습니다.
영업 메일 예제에서는 "영업 메일을 써 줘" 한 줄이면 충분했지만, 이번에는 사정이 다릅니다.

- 매칭 에이전트에 넘길 데이터: 자산 10건 + CVE 후보 약 40건(설명 포함) → 수천 토큰
- `as_tool` 을 쓰면 매니저가 이 데이터를 **자기 출력으로 다시 옮겨 적어서** 넘겨야 합니다

| 문제 | 설명 |
|---|---|
| 비용 · 속도 | 출력 토큰은 입력 토큰보다 비싸고 느림 |
| 누락 · 변형 | LLM이 옮겨 적다가 CVE를 빠뜨리거나 설명을 요약해 버릴 수 있음 |

그래서 이 노트북은 **큰 데이터는 파이썬 딕셔너리(`state`)에 두고, 매니저에게는 요약만** 보여줍니다.
매니저는 요약을 보고 **"다음에 무엇을 할지"만 결정**합니다.

> `as_tool` 도 내부적으로는 `Runner.run(하위_에이전트, input)` 을 호출하는 도구일 뿐입니다.
> 여기서는 그 포장을 직접 만들어서, 하위 에이전트가 입력을 매니저 대신 `state` 에서 읽게 합니다.

In [ ]:
# 작업 메모리: 도구들이 주고받는 큰 데이터를 보관한다. 매니저(LLM)는 이 내용을 직접 보지 않는다.
state = {}


def reset_state():
    """새 실행을 시작할 때 작업 메모리를 비웁니다."""
    state.clear()
    state.update({
        "assets": [],        # list_assets 결과
        "candidates": {},    # 제품 -> CVE 후보 목록 (search_cves)
        "matched": [],       # 실제 영향 CVE (match_cves)
        "briefings": {},     # 담당자 -> OwnerBriefing (write_briefings)
        "sent": set(),       # 이미 메일을 보낸 담당자 (중복 발송 방지)
        "skipped": {},       # 보내지 않기로 한 담당자 -> 이유 (6단계)
    })


reset_state()


# 매니저가 search_cves 를 동시에 여러 번 부를 수 있으므로, NVD 호출 간격을 코드로 지킨다.
_nvd_lock = threading.Lock()
_nvd_last_call = 0.0


def nvd_throttle():
    """직전 NVD 호출로부터 NVD_DELAY 초가 지날 때까지 기다립니다."""
    global _nvd_last_call
    with _nvd_lock:
        wait = NVD_DELAY - (time.monotonic() - _nvd_last_call)
        if wait > 0:
            time.sleep(wait)
        _nvd_last_call = time.monotonic()

## 3단계. 부품을 도구로 포장하기

매니저가 부를 도구 5개를 만듭니다. 도구를 만들 때 중요한 점은 세 가지입니다.

1. **docstring 이 곧 사용 설명서입니다.** 매니저는 코드를 보지 못합니다. 도구 이름과 docstring 만 보고 언제, 무엇을 부를지 결정합니다.
2. **반환값은 매니저가 읽을 요약입니다.** 순서가 틀렸거나 실패하면 예외를 던지는 대신 "먼저 무엇을 하라"고 문장으로 알려 줍니다. 매니저가 읽고 스스로 바로잡습니다.
3. **밖으로 나가는 행동의 안전장치는 코드에 둡니다.** LLM이 순서를 정하게 된 만큼, 프롬프트로 "조심해"라고 부탁하는 것만으로는 부족합니다.

### 조회 도구 (파이썬 코드)

`search_cves` 는 매니저가 **여러 제품을 한꺼번에(병렬로)** 호출할 수 있습니다.
SDK는 동기 함수 도구를 별도 스레드에서 실행하므로 NVD 호출이 몰릴 수 있습니다. `nvd_throttle()` 로 호출 간격을 지킵니다.

In [ ]:
def list_assets() -> str:
    """회사 자산 목록을 불러옵니다. 작업을 시작할 때 가장 먼저 호출하세요.
    각 자산의 id, 이름, 제품명(product), 버전, 담당자(owner), 중요도, 인터넷 노출 여부를 돌려줍니다.
    """
    assets = load_assets()["assets"]
    state["assets"] = assets

    summary = [
        {k: a[k] for k in ("id", "name", "product", "version", "owner", "criticality", "internet_facing")}
        for a in assets
    ]
    return json.dumps(summary, ensure_ascii=False)


def search_cves(product: str) -> str:
    """NVD에서 제품 하나의 최근 CVE 후보를 조회해 작업 메모리에 저장합니다.
    조회할 제품마다 한 번씩 호출하세요. 여러 제품을 동시에 호출해도 됩니다.

    Args:
        product: 조회할 제품 이름. list_assets 결과의 product 값을 그대로 쓰세요. (예: Apache Tomcat)
    """
    nvd_throttle()
    try:
        cves = search_cves_for_product(product)
    except requests.RequestException as e:
        return f"{product}: NVD 조회 실패 ({type(e).__name__}). 잠시 후 다시 호출해 보세요."

    state["candidates"][product] = cves
    if not cves:
        return f"{product}: 최근 CVE 후보 없음"
    ids = ", ".join(f"{c['cve_id']}(CVSS {c['cvss_score']})" for c in cves)
    return f"{product}: 후보 {len(cves)}건 저장 — {ids}"

에이전트에 붙이기 전에 **함수부터 직접 호출해 검증**합니다. (원래 노트북과 같은 원칙)

In [ ]:
print(list_assets()[:300], "...\n")
print(search_cves("Apache Tomcat"))

reset_state()   # 테스트로 채운 작업 메모리를 비운다

### 판단 도구 (에이전트를 감싼 도구)

`match_cves`, `write_briefings` 는 안에서 `Runner.run` 으로 하위 에이전트(`matcher`, `briefer`)를 실행합니다.

- 입력은 매니저에게서 받지 않고 **`state` 에서 읽습니다**
- 구조화 출력(`output_type`)은 **`state` 에 저장**합니다
- 매니저에게는 **몇 건이 남았는지, 누구에게 무엇이 갈지** 요약만 돌려줍니다

선행 단계가 안 됐으면 "먼저 무엇을 하라"는 문장을 돌려줘서, 매니저가 순서를 바로잡게 합니다.

In [ ]:
async def match_cves() -> str:
    """수집한 CVE 후보 중 우리 자산에 실제로 영향을 주는 것만 골라냅니다. (자산 매칭 에이전트 호출)
    조회할 제품의 search_cves 가 모두 끝난 뒤 한 번 호출하세요.
    """
    if not state["assets"]:
        return "자산 목록이 없습니다. 먼저 list_assets 를 호출하세요."
    if not state["candidates"]:
        return "CVE 후보가 없습니다. 먼저 search_cves 로 후보를 모으세요."

    payload = {
        "assets": [
            {k: a[k] for k in ("id", "name", "product", "version", "criticality", "internet_facing")}
            for a in state["assets"]
        ],
        "cve_candidates": state["candidates"],
    }
    m = await Runner.run(matcher, json.dumps(payload, ensure_ascii=False))

    result = m.final_output
    state["matched"] = result.matched
    state["briefings"] = {}      # 매칭이 바뀌면 이전 브리핑은 무효

    total = sum(len(v) for v in state["candidates"].values())
    head = (f"제품 {len(state['candidates'])}종, 후보 {total}건 중 "
            f"실제 영향 {len(result.matched)}건, 제외 {len(result.excluded)}건")
    if not result.matched:
        return head + " — 영향받는 취약점이 없으므로 브리핑과 메일은 필요 없습니다."
    lines = [f"- {x.cve_id} -> {x.asset_id} (CVSS {x.cvss_score}): {x.reason}" for x in result.matched]
    return head + "\n" + "\n".join(lines)


async def write_briefings() -> str:
    """확정된 취약점으로 담당자별 브리핑(위험도, 조치, 기한, 메일 제목)을 작성합니다. (브리핑 작성 에이전트 호출)
    match_cves 가 끝난 뒤 호출하세요.
    """
    if not state["matched"]:
        return "확정된 취약점이 없습니다. match_cves 를 먼저 호출했는지 확인하세요."

    asset_info = {
        a["id"]: {
            "owner": a["owner"],
            "owner_email": a["owner_email"],
            "name": a["name"],
            "criticality": a["criticality"],
            "internet_facing": a["internet_facing"],
        }
        for a in state["assets"]
    }
    b = await Runner.run(
        briefer,
        json.dumps({"matched": [x.model_dump() for x in state["matched"]], "asset_info": asset_info},
                   ensure_ascii=False),
    )

    state["briefings"] = {br.owner: br for br in b.final_output.briefings}

    lines = []
    for br in state["briefings"].values():
        counts = Counter(a.severity for a in br.advisories)
        levels = ", ".join(f"{s} {counts[s]}" for s in SEVERITY_COLOR if counts[s])
        lines.append(f"- {br.owner}: {len(br.advisories)}건 ({levels}) / 제목: {br.headline}")
    return f"담당자 {len(state['briefings'])}명의 브리핑 작성 완료\n" + "\n".join(lines)

### 발송 도구 (파이썬 코드 + 안전장치)

매니저가 넘길 수 있는 것은 **담당자 이름 하나뿐**입니다. 받는 사람 주소와 메일 본문은 코드가 정합니다.
[security_briefing_by_code.ipynb](security_briefing_by_code.ipynb) 5단계의 발송 도구와 같은 방식입니다.

| 안전장치 | 막아 주는 것 |
|---|---|
| 인자는 담당자 이름뿐 | LLM이 임의의 주소·본문으로 메일을 보내는 것 |
| 본문은 `build_html` 이 작성된 브리핑으로 생성 | LLM이 브리핑 내용을 지어내거나 바꾸는 것 |
| `ROUTE_ALL_TO_ME` | 데모 중 실제 담당자 주소로 나가는 것 |
| `state["sent"]` 기록 | 같은 담당자에게 두 번 보내는 것 |
| `DRY_RUN` | 판단 과정만 보고 싶을 때 메일이 실제로 나가는 것 |

In [ ]:
def send_briefing_email(owner: str) -> str:
    """담당자 한 명에게 write_briefings 로 작성한 브리핑 메일을 보냅니다.
    받는 주소와 메일 본문은 작성된 브리핑으로 코드가 만들므로, 담당자 이름만 넘기면 됩니다.
    담당자 한 명당 한 번만 호출하세요.

    Args:
        owner: 담당자 이름. write_briefings 결과에 나온 이름 그대로 쓰세요. (예: 김운영)
    """
    br = state["briefings"].get(owner)
    if br is None:
        available = ", ".join(state["briefings"]) or "없음 (write_briefings 를 먼저 호출하세요)"
        return f"'{owner}' 의 브리핑이 없습니다. 보낼 수 있는 담당자: {available}"
    if owner in state["sent"]:
        return f"{owner} 에게는 이미 발송했습니다. 다시 보내지 않습니다."

    to = ROUTE_ALL_TO_ME or br.owner_email
    if DRY_RUN:
        result = f"[DRY_RUN] 발송 생략 -> {to} / 제목: {br.headline}"
    else:
        result = send_email(to, br.headline, build_html(br))

    state["sent"].add(owner)
    return f"{owner}: {result}"

### 도구 목록

`function_tool(함수)` 로 감싸면 SDK가 **함수 이름 · docstring · 타입 힌트**로 도구 스키마를 만듭니다.
`@function_tool` 데코레이터와 같은 것입니다. 여기서는 위에서 함수를 직접 테스트하려고 나중에 감쌌습니다.

In [ ]:
tools = [
    function_tool(list_assets),
    function_tool(search_cves),
    function_tool(match_cves),
    function_tool(write_briefings),
    function_tool(send_briefing_email),
]

for t in tools:
    print(f"{t.name:<20} {t.description.splitlines()[0]}")

In [ ]:
# 매니저(LLM)가 보는 search_cves 도구의 입력 스키마
tools[1].params_json_schema

## 4단계. 매니저 에이전트

참고 노트북의 Sales Manager 와 같은 구조입니다. `instructions` 에는 **역할과 규칙**, `task` 에는 **이번에 할 일**을 적습니다.

task 에 순서를 적어 주지만, 도구를 **언제 · 몇 번 · 어떤 인자로** 부를지는 매니저가 정합니다.
예를 들어 제품 10개를 한 번에 병렬로 조회할지, 하나씩 조회할지는 매니저의 판단입니다.

In [ ]:
instructions = """
당신은 한빛테크 주식회사의 보안 브리핑 매니저입니다.
도구를 사용해 오늘의 보안 취약점 브리핑을 담당자에게 전달하는 것이 목표입니다.

규칙:
- 큰 데이터는 도구가 작업 메모리에 저장합니다. 당신은 도구가 돌려주는 요약을 보고 다음에 할 일을 결정합니다.
- CVE, 위험도, 담당자 정보를 스스로 지어내지 마세요. 도구 결과만 근거로 삼으세요.
- 도구가 실패했거나 선행 단계가 필요하다고 알려 주면, 그 안내에 따라 대응하세요.
- 같은 담당자에게 메일을 두 번 보내지 마세요.

작업이 끝나면 무엇을 했는지 한국어로 짧게 보고하세요.
(조회한 제품 수, 확정된 취약점 수, 메일을 보낸 담당자와 결과, 보내지 않은 담당자가 있다면 그 이유)
"""

task = """
오늘의 보안 브리핑을 진행하세요.

1. 자산 확인: list_assets 로 자산 목록을 불러오세요.
2. CVE 수집: 자산에 있는 모든 제품에 대해 search_cves 를 호출하세요. 제품마다 한 번씩, 빠짐없이.
3. 자산 매칭: 모든 제품의 수집이 끝난 뒤 match_cves 를 한 번 호출하세요.
4. 브리핑 작성: 확정된 취약점이 있으면 write_briefings 를 호출하세요.
5. 발송: 브리핑이 작성된 담당자마다 send_briefing_email 을 한 번씩 호출하세요.
"""

security_manager = Agent(
    name="보안 브리핑 매니저",
    instructions=instructions,
    tools=tools,
    model=MODEL_NAME,
)

### 실행

`max_turns` 는 매니저가 LLM을 호출할 수 있는 최대 횟수입니다. 기본값 10은 이 작업에 빠듯하므로 30으로 늘립니다.

**실행하면 실제로 메일이 나갑니다** (`ROUTE_ALL_TO_ME` 에 따라 본인 주소로만).
판단 과정만 보려면 먼저 `DRY_RUN = True` 로 바꾸세요.

In [ ]:
reset_state()

with trace("보안 브리핑 매니저"):
    result = await Runner.run(security_manager, task, max_turns=30)

print(result.final_output)

### 매니저가 실제로 한 일

코드 오케스트레이션에서는 코드를 읽으면 순서를 알 수 있었지만, 이제는 **실행 결과를 봐야** 압니다.
`result.new_items` 에 매니저의 도구 호출과 결과가 순서대로 들어 있습니다.

"판단" 한 번은 LLM 호출 한 번입니다. 한 번의 판단에서 도구를 여러 개 부르면 **병렬로 실행**됩니다.
(`show_run` 은 6단계의 핸드오프도 보여 주도록 만들어 두었습니다.)

In [ ]:
def show_run(result):
    """에이전트가 어떤 도구를 어떤 순서로 불렀는지 출력합니다. (핸드오프 포함)"""
    outputs = {}
    for item in result.new_items:
        if item.type == "tool_call_output_item":
            raw = item.raw_item
            call_id = raw["call_id"] if isinstance(raw, dict) else raw.call_id
            outputs[call_id] = str(item.output)

    calls = ("tool_call_item", "handoff_call_item")
    decision, prev_type, n_calls = 0, None, 0
    for item in result.new_items:
        if item.type in calls:
            if prev_type not in calls:             # 연속된 호출 = 한 번의 판단에서 병렬 호출
                decision += 1
                print(f"--- 판단 {decision}: {item.agent.name} ---")
            call = item.raw_item
            args = "" if call.arguments in ("", "{}") else call.arguments
            print(f"  {call.name}({args})")
            if item.type == "tool_call_item":
                n_calls += 1
                out = outputs.get(call.call_id, "")
                print(f"      -> {out.splitlines()[0][:90] if out else '(결과 없음)'}")
        elif item.type == "handoff_output_item":
            print(f"      -> 핸드오프: {item.source_agent.name} → {item.target_agent.name} (제어가 넘어감)")
        prev_type = item.type

    print(f"\nLLM 호출 {len(result.raw_responses)}회 (매칭·브리핑 에이전트 제외) / 도구 호출 {n_calls}회 / "
          f"마지막 에이전트: {result.last_agent.name}")


show_run(result)

In [ ]:
print(f"확정 취약점 {len(state['matched'])}건 / 브리핑 {len(state['briefings'])}명 / "
      f"발송 완료: {', '.join(sorted(state['sent'])) or '없음'}\n")

for br in state["briefings"].values():
    print(f"=== {br.owner} <{br.owner_email}> — {len(br.advisories)}건")
    print(f"    제목: {br.headline}")
    for a in br.advisories:
        print(f"    [{a.severity:<3}] {a.cve_id:<16} {a.asset_name:<18} {a.deadline}")
    print()

## 5단계. 요청만 바꿔 보기

LLM 오케스트레이션의 진짜 장점은 **코드를 고치지 않고 요청만 바꿔서 다른 흐름을 만들 수 있다**는 것입니다.

"오늘은 인터넷에 노출된 자산만 보고, 긴급 건이 있는 담당자에게만 메일을 보내라"는 요청이 왔다고 합시다.

| | 해야 할 일 |
|---|---|
| by Code | 제품 필터링 코드와 발송 조건 코드를 새로 작성 |
| by LLM | 같은 도구, 같은 매니저에게 **task 문장만** 바꿔서 전달 |

이번에는 `DRY_RUN = True` 로 실제 발송 없이 매니저의 판단만 확인합니다.
매니저가 **어떤 제품을 조회에서 뺐는지, 누구에게 보내지 않았는지** 살펴보세요.

In [ ]:
task2 = """
오늘은 인터넷에 노출된 자산만 점검하세요.

1. list_assets 로 자산을 확인하고, internet_facing 이 true 인 자산의 제품만 search_cves 로 조회하세요.
2. match_cves, write_briefings 로 브리핑을 만드세요.
3. '긴급' 건이 있는 담당자에게만 send_briefing_email 을 호출하세요.
   메일을 보내지 않은 담당자가 있으면 보고에 그 이유를 적으세요.
"""

prev_dry_run, DRY_RUN = DRY_RUN, True     # 이번 실행은 판단만 보고 실제로는 보내지 않는다
reset_state()
try:
    with trace("보안 브리핑 매니저 - 인터넷 노출 자산만"):
        result2 = await Runner.run(security_manager, task2, max_turns=30)
finally:
    DRY_RUN = prev_dry_run                 # 원래 설정으로 되돌린다

show_run(result2)
print("\n[ 매니저 보고 ]")
print(result2.final_output)

## 6단계. Orchestrating by LLMs — via Handoffs

[openai_agent_code.ipynb](openai_agent_code.ipynb) 의 **Step 9: 핸드오프** 방식입니다.

4~5단계의 매니저는 모든 도구를 직접 불렀고, 도구가 끝나면 **제어가 항상 매니저에게 돌아왔습니다.**
핸드오프는 한 에이전트가 다른 에이전트에게 **대화 자체를 넘기는** 방식입니다. 넘긴 쪽으로는 돌아오지 않습니다.

```
도구 (4~5단계)    매니저 → 도구 → 매니저 → 도구 → ... → 매니저가 최종 보고          A → B → A
핸드오프 (6단계)  분석 매니저 → (분석 끝) → 메일 발송 에이전트가 이어받아 발송·보고     A → B
```

| | 도구 (as tool / function tool) | 핸드오프 |
|---|---|---|
| 제어 | 결과를 받아 **호출한 에이전트로 돌아옴** | **받는 에이전트로 넘어가고** 돌아오지 않음 |
| 받는 쪽이 보는 것 | 호출한 쪽이 만든 인자 | **지금까지의 대화 전체** (요청, 도구 호출과 결과) |
| 최종 답변 | 호출한 에이전트 | 넘겨받은 에이전트 (`result.last_agent`) |
| 어울리는 경우 | 계획 · 취합 (Planning Agent) | 역할이 나뉜 업무 이관 (분석 담당 → 발송 담당) |

### 역할 나누기

| 에이전트 | 도구 | 하는 일 |
|---|---|---|
| 보안 분석 매니저 | `list_assets`, `search_cves`, `match_cves`, `write_briefings` | 분석하고 브리핑을 작성한 뒤 **발송을 넘김** |
| 메일 발송 에이전트 | `send_briefing_email` | 넘겨받은 대화에서 담당자와 발송 조건을 확인하고 발송 |

**분석 매니저에게는 발송 도구가 아예 없습니다.** 분석하는 에이전트는 메일을 보낼 수 없다는 것 자체가 안전장치입니다.

> 참고 노트북의 설명처럼, SDK는 핸드오프를 내부적으로 **도구**로 구현합니다.
> 매니저에게는 `transfer_to_<에이전트 이름>` 이라는 도구가 보이고, 매니저가 이 도구를 부르면 SDK가 다음 턴부터 받는 에이전트를 실행합니다.

### 메일 발송 에이전트 (넘겨받는 쪽)

네 가지를 챙깁니다.

1. **핸드오프 도구 이름은 영문으로 지정합니다.** 도구 이름은 에이전트 이름으로 만들어지는데,
   `"메일 발송 에이전트"` 처럼 한글이면 `transfer_to___________` 로 깨집니다. (OpenAI 함수 이름은 영문 · 숫자 · 밑줄만 가능)
   그래서 `handoff(에이전트, tool_name_override=...)` 로 이름을 직접 줍니다.
2. **`handoff_description`** 은 매니저가 "언제 넘길지" 판단할 때 보는 설명입니다. 핸드오프 도구의 설명에 들어갑니다.
3. **`RECOMMENDED_PROMPT_PREFIX`** 는 SDK가 권장하는 지시문 머리말입니다.
   "여러 에이전트가 협업하는 시스템의 일부이며, 핸드오프는 `transfer_to_` 도구로 한다"는 설명이 들어 있습니다.

4. **보내지 않는 결정도 도구로 기록합니다.** 발송 도구와 함께 `skip_briefing_email(owner, reason)` 도구를 줍니다.
   그래서 참고 노트북의 `sales_sender` 처럼 `tool_choice="required"` 로 도구 호출을 강제해도,
   조건에 맞지 않는 담당자에게 **억지로 메일이 나가지 않습니다.** (보내지 않는 것도 도구 호출이므로)

> 이 노트북을 만들며 테스트할 때, `tool_choice` 없이 실행했더니 발송 에이전트가
> **도구를 한 번도 부르지 않고 "발송 완료"라고 보고**한 적이 있습니다. (실제 발송 0건)
> 그래서 모든 담당자에 대해 발송 또는 미발송을 **도구로 남기게** 하고, 결과는 LLM의 보고가 아니라 `state` 로 확인합니다.

In [ ]:
def skip_briefing_email(owner: str, reason: str) -> str:
    """발송 조건에 맞지 않아 이 담당자에게는 메일을 보내지 않기로 한 결정을 기록합니다.

    Args:
        owner: 담당자 이름. write_briefings 결과에 나온 이름 그대로 쓰세요. (예: 박데이터)
        reason: 보내지 않는 이유 한 문장 (예: 긴급·높음 건이 없음)
    """
    if owner not in state["briefings"]:
        available = ", ".join(state["briefings"]) or "없음"
        return f"'{owner}' 의 브리핑이 없습니다. 담당자: {available}"
    if owner in state["sent"]:
        return f"{owner} 에게는 이미 발송했습니다. 미발송으로 바꿀 수 없습니다."

    state["skipped"][owner] = reason
    return f"{owner}: 발송 안 함 — {reason}"


mail_sender = Agent(
    name="메일 발송 에이전트",
    handoff_description="담당자별 브리핑 작성이 끝난 뒤, 메일 발송을 맡는 에이전트",
    instructions=RECOMMENDED_PROMPT_PREFIX + """
당신은 보안 브리핑 메일 발송 담당자입니다. 보안 분석 매니저에게서 업무를 넘겨받습니다.

- 대화 기록에서 write_briefings 결과를 찾아, 브리핑이 작성된 담당자와 위험도별 건수를 확인하세요.
  위험도는 메일 제목이 아니라 괄호 안의 위험도별 건수로 판단하세요.
- 사용자의 요청에 발송 조건이 있으면 그 조건을 따르세요.
- 브리핑이 작성된 담당자 '모두'에 대해 둘 중 하나를 반드시 호출하세요.
  - 조건에 맞으면 send_briefing_email (메일 주소와 본문은 도구가 만듭니다)
  - 조건에 맞지 않으면 skip_briefing_email (reason 에는 그 담당자의 위험도별 건수를 함께 적으세요)
- 도구를 호출하지 않고 발송했다고 보고하면 안 됩니다. 보고는 도구 결과만 근거로 하세요.

끝나면 한국어로 짧게 보고하세요. (발송한 담당자와 결과, 보내지 않은 담당자와 그 이유)
""",
    tools=[function_tool(send_briefing_email), function_tool(skip_briefing_email)],
    model=MODEL_NAME,
    model_settings=ModelSettings(tool_choice="required"),   # 첫 응답에서 반드시 도구를 호출
)

# 한글 에이전트 이름으로는 도구 이름이 깨지므로 영문 이름을 직접 준다
to_mail_sender = handoff(mail_sender, tool_name_override="transfer_to_mail_sender")

print("핸드오프 도구 이름:", to_mail_sender.tool_name)
print("핸드오프 도구 설명:", to_mail_sender.tool_description)

### 보안 분석 매니저 (넘기는 쪽)

4단계의 매니저에서 **발송 도구를 빼고**, 대신 `handoffs` 에 메일 발송 에이전트를 넣습니다.
참고 노트북의 `Agent(..., tools=tools, handoffs=handoffs)` 와 같은 구조입니다.

task 의 5번에 **발송 조건**('긴급' 또는 '높음' 건이 있는 담당자만)을 적었습니다.
이 조건은 매니저에게만 말했지만, 메일 발송 에이전트는 핸드오프로 **대화 전체를 넘겨받기 때문에** 조건을 알 수 있습니다.

In [ ]:
analysis_tools = [t for t in tools if t.name != "send_briefing_email"]

analysis_instructions = RECOMMENDED_PROMPT_PREFIX + """
당신은 한빛테크 주식회사의 보안 분석 매니저입니다.
도구로 취약점을 분석해 담당자별 브리핑을 작성한 뒤, 메일 발송은 메일 발송 에이전트에게 넘깁니다.
당신에게는 메일 발송 도구가 없습니다.

규칙:
- 큰 데이터는 도구가 작업 메모리에 저장합니다. 당신은 도구가 돌려주는 요약을 보고 다음에 할 일을 결정합니다.
- CVE, 위험도, 담당자 정보를 스스로 지어내지 마세요. 도구 결과만 근거로 삼으세요.
- 도구가 실패했거나 선행 단계가 필요하다고 알려 주면, 그 안내에 따라 대응하세요.
- 확정된 취약점이 없으면 넘기지 말고, 그 사실을 보고하고 끝내세요.
"""

handoff_task = """
오늘의 보안 브리핑을 진행하세요.

1. 자산 확인: list_assets 로 자산 목록을 불러오세요.
2. CVE 수집: 자산에 있는 모든 제품에 대해 search_cves 를 호출하세요. 제품마다 한 번씩, 빠짐없이.
3. 자산 매칭: 모든 제품의 수집이 끝난 뒤 match_cves 를 한 번 호출하세요.
4. 브리핑 작성: 확정된 취약점이 있으면 write_briefings 를 호출하세요.
5. 발송: 메일 발송 에이전트에게 넘기세요.
   발송 조건은 '긴급' 또는 '높음' 건이 하나라도 있는 담당자에게만 보내는 것입니다.
"""

analysis_manager = Agent(
    name="보안 분석 매니저",
    instructions=analysis_instructions,
    tools=analysis_tools,
    handoffs=[to_mail_sender],
    model=MODEL_NAME,
)

print("분석 매니저가 보는 도구:", [t.name for t in analysis_tools] + [to_mail_sender.tool_name])

### 실행

4단계에서 이미 메일을 보냈으므로 이번에는 `DRY_RUN = True` 로 판단 과정만 봅니다.

`show_run` 의 "판단" 제목에 **어느 에이전트가 판단했는지** 나옵니다.
`transfer_to_mail_sender` 호출 뒤로는 판단 주체가 메일 발송 에이전트로 바뀌는 것을 확인하세요.

In [ ]:
prev_dry_run, DRY_RUN = DRY_RUN, True     # 4단계에서 이미 보냈으므로 이번에는 판단만 본다
reset_state()
try:
    with trace("보안 브리핑 - 핸드오프"):
        result3 = await Runner.run(analysis_manager, handoff_task, max_turns=30)
finally:
    DRY_RUN = prev_dry_run                 # 원래 설정으로 되돌린다

show_run(result3)
print(f"\n[ {result3.last_agent.name} 의 최종 보고 ]")
print(result3.final_output)

### 핸드오프가 실제로 일어났는지 코드로 확인

참고 노트북의 작성자는 핸드오프를 "신뢰하기 어렵다(unreliable)"고 평가합니다.
**넘길지 말지를 LLM이 결정**하기 때문입니다. 매니저가 넘기지 않고 스스로 끝내 버리면 메일은 한 통도 나가지 않습니다.
넘겨받은 에이전트가 도구를 부르지 않고 "보냈다"고 보고할 수도 있습니다.

그래서 LLM의 보고가 아니라 **`state` 에 남은 기록**으로 확인합니다.
브리핑이 작성된 담당자는 모두 '발송' 또는 '발송 안 함(이유)' 중 하나로 기록되어 있어야 합니다.

발송 조건처럼 **규칙으로 판단할 수 있는 것**은 코드로 다시 계산해 LLM의 결정과 비교합니다.
실제로 테스트 중에 발송 에이전트가 "높음 건은 있지만 ... 조건 충족됨" 이라는 **앞뒤가 맞지 않는 이유로 발송을 건너뛴** 적이 있습니다.
결정 이유가 도구로 기록되어 있었기 때문에 이 오류를 찾을 수 있었습니다.

> 실무라면 이런 규칙은 처음부터 코드로 처리하는 것이 맞습니다.
> ([security_briefing_by_code.ipynb](security_briefing_by_code.ipynb) 의 원칙: 판단은 모델에게, 계산과 조회는 코드에게)
> 여기서는 핸드오프로 **대화(발송 조건)가 함께 넘어가는 것**을 보여 주려고 LLM에게 맡겼습니다.

In [ ]:
handed_off = result3.last_agent is mail_sender
decided = state["sent"] | set(state["skipped"])
undecided = sorted(set(state["briefings"]) - decided)

print(f"핸드오프: {'됨' if handed_off else '안 됨 — 매니저가 넘기지 않고 끝냈습니다'}")
print(f"브리핑 {len(state['briefings'])}명 / 발송 {len(state['sent'])}명: "
      f"{', '.join(sorted(state['sent'])) or '없음'}")
for owner, reason in state["skipped"].items():
    print(f"  발송 안 함: {owner} — {reason}")
if undecided:
    print(f"기록 없는 담당자: {', '.join(undecided)} — 에이전트가 도구를 부르지 않았습니다. "
          f"보고를 믿지 말고 다시 실행하세요.")

# 발송 조건('긴급' 또는 '높음' 건이 하나라도 있음)을 코드로 다시 계산해 LLM의 결정과 비교한다
print("\n[ 발송 조건 검사 ]")
for owner, br in state["briefings"].items():
    should_send = any(a.severity in ("긴급", "높음") for a in br.advisories)
    did_send = owner in state["sent"]
    verdict = "OK" if should_send == did_send else "불일치"
    print(f"  [{verdict}] {owner}: 조건 {'충족' if should_send else '미충족'} → "
          f"{'발송' if did_send else '발송 안 함'}")

## 정리

### 이 예제에서 확인한 것

| 항목 | 내용 |
|---|---|
| 같은 부품, 다른 지휘자 | security_briefing_by_code 의 함수와 에이전트를 그대로 도구로 포장 |
| 매니저가 흐름을 결정 | 병렬 조회, 선행 단계 확인, 발송 대상 선택을 LLM이 판단 |
| 작업 메모리(`state`) | 큰 데이터는 코드가 보관하고 매니저는 요약만 보고 결정 → 비용 절감, 누락 방지 |
| 코드에 둔 안전장치 | 수신자 라우팅, 코드로 만드는 본문, 중복 발송 방지, `DRY_RUN` |
| 요청만 바꿔 재사용 | 코드 수정 없이 "인터넷 노출 자산만 / 긴급 건만" 흐름 실행 |
| 핸드오프 | 분석 매니저가 메일 발송 에이전트에게 제어를 넘김. 발송 도구는 발송 에이전트만 가짐 |
| 결정도 도구로 기록 | 발송 · 미발송 모두 도구로 남겨 LLM의 보고 대신 `state` 로 검증. 규칙(발송 조건)은 코드로 재계산 |

### 언제 어느 쪽을 쓰나

| 상황 | 추천 |
|---|---|
| 매일 같은 순서로 도는 정형 작업 (예: 매일 아침 브리핑) | **by Code** — 싸고 예측 가능 |
| 요청이 매번 다르고, 중간 결과에 따라 다음 행동이 바뀌는 작업 | **by LLM** |
| 실무 | 둘을 섞음 — 정해진 파이프라인은 코드로, 분기 판단만 LLM에게 |

### 도구냐 핸드오프냐

| 상황 | 추천 |
|---|---|
| 여러 결과를 모아 한 에이전트가 최종 판단 · 검증해야 함 | **도구** — 제어가 돌아오므로 취합할 수 있음 |
| 업무가 담당 에이전트에게 이관되고, 넘긴 뒤에는 관여할 필요 없음 | **핸드오프** |
| 반드시 실행되어야 하는 단계 (예: 발송) | 핸드오프만 믿지 말고 **코드로 확인**하거나 by Code 로 |

### 비용

by Code 는 LLM 호출이 3회(매칭, 브리핑, 발송)였습니다. by LLM 은 매칭 · 브리핑 2회에 `show_run` 이 보여준 **LLM 호출 횟수**가 더해집니다.
(4~5단계는 매니저가 발송 도구를 직접 부르고, 6단계는 메일 발송 에이전트의 호출이 이 횟수에 포함됩니다.)
매니저는 호출될 때마다 지금까지의 대화와 도구 결과 전체를 다시 입력으로 받습니다. 핸드오프로 넘겨받은 에이전트도 마찬가지입니다.
그래서 **도구 결과를 짧은 요약으로 돌려주는 것이 비용을 줄이는 핵심**입니다.
자세한 내용은 [context_token_num.md](../contextw/context_token_num.md) 를 참고하세요.

### Trace

https://platform.openai.com/traces 에서 **매니저 → 도구 → 하위 에이전트** 호출이 하나의 트레이스 안에 중첩되어 보입니다.
6단계 트레이스에서는 **분석 매니저 → 메일 발송 에이전트** 로 핸드오프되는 지점도 확인할 수 있습니다.
"왜 이 담당자에게는 메일을 보내지 않았는가" 같은 질문에 답할 근거가 됩니다.